# CD-MSC — Approximate HPSS + Histogram Matching (Exp 7)

Builds on **Exp 4** (C-DANN + domain-balanced batches, BAunseen=0.2626) with two additions applied in `dataset.py` at training time — no feature re-extraction needed.

**Approximate HPSS**: Wiener masking on the log-mel spectrogram suppresses broadband percussive noise while preserving tonal (harmonic) wingbeat content.  
**Histogram matching**: For D5 training clips, randomly re-scales each mel bin to match the mean/std of a randomly chosen field domain (D1–D4). Gives the bottleneck species (An. dirus, An. minimus, An. stephensi) synthetic field-domain training data.

**Before running:** Runtime → Change runtime type → A100 GPU (Colab Pro)

In [ ]:
# Confirm GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — check Runtime settings')

In [ ]:
# Clone repo if needed, then pull latest and install deps
import os
if not os.path.exists('/content/CD-MSC'):
    !git clone https://github.com/saha23s/CD-MSC.git /content/CD-MSC
%cd /content/CD-MSC
!git checkout aaron/preprocessing
!git pull origin aaron/preprocessing
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Restore pre-extracted features from Drive
import shutil, pathlib

src = pathlib.Path('/content/drive/MyDrive/CD-MSC-feature')
dst = pathlib.Path('Development_data/feature')
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    pkls = list(dst.glob('*.pkl'))
    print(f'Features restored: {len(pkls)} pkl files.')
    stats_json = dst / 'domain_feature_stats.json'
    print(f'Domain stats present: {stats_json.exists()}')
else:
    print('ERROR: MyDrive/CD-MSC-feature not found on Drive.')
    print('Run colab_quickstart.ipynb first to extract and back up features.')

In [ ]:
# Compute per-domain, per-mel-bin statistics from the training pkl
# Run this once — skips if domain_feature_stats.json already exists locally.
import pickle, json, numpy as np
from collections import defaultdict
from pathlib import Path

stats_path = Path('Development_data/feature/domain_feature_stats.json')

if stats_path.exists():
    print(f'domain_feature_stats.json already exists — skipping computation.')
    print('Delete the file and re-run this cell if you want to recompute.')
else:
    train_pkl = next(Path('Development_data/feature').glob('training_*.pkl'), None)
    if train_pkl is None:
        raise FileNotFoundError('No training_*.pkl found in Development_data/feature/')
    print(f'Reading {train_pkl} ...')
    with open(train_pkl, 'rb') as f:
        payload = pickle.load(f)

    domain_frames = defaultdict(list)
    for item in payload['items']:
        feat = item['feature'].astype(np.float32)  # [T, 64], unnormalized log-mel
        domain_frames[item['domain_label']].append(feat)

    domain_names = {0: 'D1', 1: 'D2', 2: 'D3', 3: 'D4', 4: 'D5'}
    stats = {}
    for label, name in domain_names.items():
        all_frames = np.concatenate(domain_frames[label], axis=0)  # [N_frames, 64]
        stats[name] = {
            'mean': all_frames.mean(axis=0).tolist(),
            'std':  all_frames.std(axis=0).tolist(),
        }
        print(f'{name}: {len(domain_frames[label])} clips, {all_frames.shape[0]} frames')

    stats_path.write_text(json.dumps(stats, indent=2))
    print(f'Saved to {stats_path}')

    # Back up to Drive alongside the pkl files
    drive_stats = Path('/content/drive/MyDrive/CD-MSC-feature/domain_feature_stats.json')
    import shutil
    shutil.copy(stats_path, drive_stats)
    print(f'Backed up to Drive: {drive_stats}')

In [ ]:
# ── edit these ───────────────────────────────────────────────────────────────
USE_APPROX_HPSS = True    # Wiener masking on log-mel to suppress percussive noise
HIST_MATCH      = True    # Shift D5 clip distribution to random field domain at training time
SEED            = 42
DANN_ALPHA_MAX  = 0.3
CDANN           = True
BALANCE_BATCHES = True
BALANCE_MODE    = "domain"
BATCH_SIZE      = 64
SUPCON_WEIGHT   = 0.0
SPEC_AUGMENT    = False
CMN             = False
D5_NOISE_STD    = 0.0
USE_DELTA       = False
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Write config for this experiment
import json

with open('configs/default_experiment.json') as f:
    cfg = json.load(f)

cfg['seed']               = SEED
cfg['use_approx_hpss']    = USE_APPROX_HPSS
cfg['hist_match']         = HIST_MATCH
cfg['dann_alpha_max']     = DANN_ALPHA_MAX
cfg['cdann']              = CDANN
cfg['batch_balance_domain'] = BALANCE_BATCHES
cfg['balance_mode']       = BALANCE_MODE
cfg['batch_size']         = BATCH_SIZE
cfg['supcon_weight']      = SUPCON_WEIGHT
cfg['spec_augment']       = SPEC_AUGMENT
cfg['cmn']                = CMN
cfg['d5_noise_std']       = D5_NOISE_STD
cfg['use_delta']          = USE_DELTA

dann_tag     = f'_dann{DANN_ALPHA_MAX}' if DANN_ALPHA_MAX > 0 else ''
cdann_tag    = '_cdann' if CDANN else ''
balanced_tag = '_balanced' if BALANCE_BATCHES else ''
hpss_tag     = '_hpss' if USE_APPROX_HPSS else ''
hm_tag       = '_histmatch' if HIST_MATCH else ''

config_path = f'configs/histmatch_seed{SEED}{dann_tag}{cdann_tag}{balanced_tag}{hpss_tag}{hm_tag}.json'
with open(config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Config written: {config_path}')
print(f'  seed           = {SEED}')
print(f'  use_approx_hpss = {USE_APPROX_HPSS}')
print(f'  hist_match     = {HIST_MATCH}')
print(f'  dann_alpha_max = {DANN_ALPHA_MAX}')
print(f'  cdann          = {CDANN}')
print(f'  balance_batches = {BALANCE_BATCHES} ({BALANCE_MODE})')
print(f'  batch_size     = {BATCH_SIZE}')
print()
print('Output dir will be:')
print(f'  outputs/MTRCNN_seed{SEED}_B{BATCH_SIZE}_E100_earlystop_min20_pati10{dann_tag}{cdann_tag}{balanced_tag}{hpss_tag}{hm_tag}/')

## Train

Expect ~30–60 min on A100. Early stopping monitors mean field-domain BA (D1–D4), min epoch 20, patience 10.

Watch that training loss decreases normally — HPSS and histogram matching add per-sample compute but should not destabilise training.

In [ ]:
!python train.py --config $config_path

In [ ]:
# Save outputs to Drive before session expires
import shutil
shutil.copytree('outputs', '/content/drive/MyDrive/CD-MSC-outputs', dirs_exist_ok=True)
print('Outputs saved to Google Drive.')

## Results

Compares all runs found in `outputs/` against the released baseline and Exp 4.

In [ ]:
import json, pathlib

def load_metrics(output_dir):
    p = pathlib.Path(output_dir) / 'best_model_eval' / 'test_metrics.json'
    return json.loads(p.read_text()) if p.exists() else None

rows = []
rows.append({'run': 'Baseline 10-seed mean (released)', 'BA_seen': 0.8806, 'BA_unseen': 0.1751, 'DSG': 0.7055})
rows.append({'run': 'Exp 4 — C-DANN + balanced (best so far)', 'BA_seen': 0.7950, 'BA_unseen': 0.2626, 'DSG': 0.5324})

baseline_dir = 'MTRCNN_seed42_B64_E100_earlystop_min10_pati5'
for p in sorted(pathlib.Path('outputs').glob('*/best_model_eval/test_metrics.json')):
    run_name = p.parts[-3]
    if run_name == baseline_dir:
        continue
    m = json.loads(p.read_text())
    if m.get('BA_seen') is None:
        continue
    rows.append({'run': run_name, 'BA_seen': m['BA_seen'], 'BA_unseen': m['BA_unseen'], 'DSG': m['DSG']})

if not rows:
    print('No results found yet — run training first.')
else:
    w = 75
    print(f"{'Run':<{w}} {'BA_seen':>8} {'BA_unseen':>10} {'DSG':>8}")
    print('-' * (w + 30))
    for r in rows:
        print(f"{r['run']:<{w}} {r['BA_seen']:>8.4f} {r['BA_unseen']:>10.4f} {r['DSG']:>8.4f}")